In [1]:
import pandas as pd
import sqlite3
import plotly.graph_objects as go
import numpy as np

conn = sqlite3.connect('../data/checking-logs.sqlite')

In [2]:
query = """
SELECT uid, timestamp
FROM checker
WHERE uid LIKE 'user_%'
  AND labname = 'project1'
  AND status = 'ready'
"""

In [3]:
df = pd.read_sql_query(query, conn)

df['timestamp'] = pd.to_datetime(df['timestamp'])
df['date'] = df['timestamp'].dt.date


In [4]:
daily = df.groupby(['date', 'uid']).size().unstack(fill_value=0)
cumulative = daily.cumsum()

In [5]:
users = cumulative.columns
steps = np.arange(1, len(cumulative) + 1)

In [6]:
fig = go.Figure()

for user in users:
    fig.add_trace(
        go.Scatter(
            x=steps,
            y=cumulative[user],
            mode='lines+markers',
            name=user
        )
    )

In [7]:
frames = []
for i in range(len(steps)):
    frames.append(
        go.Frame(
            data=[
                go.Scatter(
                    x=steps[:i+1],
                    y=cumulative[user].iloc[:i+1]
                )
                for user in users
            ],
            name=str(i)
        )
    )

fig.frames = frames

fig.update_layout(
    title="Dynamic of commits per user in project1",
    xaxis_title="Step",
    yaxis_title="Commits",
    updatemenus=[
        {
            "type": "buttons",
            "showactive": False,
            "buttons": [
                {
                    "label": "play",
                    "method": "animate",
                    "args": [
                        None,
                        {
                            "frame": {"duration": 300, "redraw": True},
                            "fromcurrent": True,
                            "transition": {"duration": 0}
                        }
                    ]
                }
            ]
        }
    ],
    width=1200,
    height=600
)

fig.show()
conn.close()